In [9]:
import numpy as np
from sklearn.linear_model import LinearRegression
import unittest
from matrix_helper import *

#### Phương pháp bình phương tối tiểu - OLS

OLS tìm $\hat{\boldsymbol{\beta}}$ tối thiểu hóa tổng bình phương phần dư (Residual Sum of Squares):
$$RSS(\boldsymbol{\beta}) = \|\mathbf{y} - \mathbf{X}\boldsymbol{\beta}\|_2^2 = \sum_{i=1}^{n}(y_i - \mathbf{x}_i^T\boldsymbol{\beta})^2.$$

Nếu $\mathbf{X}^T\mathbf{X}$ khả nghịch, nghiệm OLS duy nhất được xác định bởi:

$$\hat{\boldsymbol{\beta}}_{OLS} = (\mathbf{X}^T\mathbf{X})^{-1}\mathbf{X}^T\mathbf{y}.$$

Ước lượng phương sai nhiễu:
$$\hat{\sigma}^2 = \frac{RSS}{n - p - 1} = \frac{\\|\mathbf{y} - \mathbf{X}\hat{\boldsymbol{\beta}}\\|^2}{n - p - 1}.$$

In [10]:
# --- Unittest cho ham ols_fit ---

from ols_implementation import ols_fit

class TestOLSFitAgainstLibraries(unittest.TestCase):
    
    def setUp(self):
        # Tao bo du lieu gia lap co nhieu
        # X bao gom cot intercept la 1.0
        self.X_list = [
            [1.0, 2.5, 1.2],
            [1.0, 1.2, 0.8],
            [1.0, 3.8, 2.5],
            [1.0, 4.1, 3.1],
            [1.0, 5.5, 4.0],
            [1.0, 0.5, 0.1]
        ]
        
        # y la vector cot 2D nghiem ngat
        self.y_list = [
            [7.2],
            [4.5],
            [10.1],
            [11.0],
            [14.8],
            [2.1]
        ]
        
        self.X_arr = np.array(self.X_list)
        self.y_arr = np.array(self.y_list)

    def test_ols_against_sklearn(self):
        """
        Verify custom ols_fit beta coefficients and variance against scikit-learn.
        """
        # Lay ket qua tu custom
        custom_beta, custom_sigma_sqr = ols_fit(self.X_list, self.y_list)
        custom_beta_flat = [b[0] for b in custom_beta] # Lam phang ve 1D de so sanh
        
        # Lay ket qua tu sklearn
        # fit_intercept=False vi X_list da co cot 1s o dau
        model = LinearRegression(fit_intercept=False)
        model.fit(self.X_arr, self.y_arr)
        
        # model.coef_ co dang (1, p) khi y co dang (n, 1)
        sklearn_beta_flat = model.coef_[0] 
        
        # Kiem tra he so Beta khop toi 5 chu so thap phan
        np.testing.assert_almost_equal(custom_beta_flat, sklearn_beta_flat, decimal=5)
        
        # Tinh sigma binh phuong tuong duong cua sklearn de so sanh
        sklearn_predictions = model.predict(self.X_arr)
        sklearn_rss = np.sum((self.y_arr - sklearn_predictions) ** 2)
        n = len(self.X_list)
        p_plus_1 = len(self.X_list[0])
        sklearn_sigma_sqr = sklearn_rss / (n - p_plus_1)
        
        # Kiem tra sigma binh phuong khop nhau
        self.assertAlmostEqual(custom_sigma_sqr, sklearn_sigma_sqr, places=5)

    def test_ols_against_numpy_lstsq(self):
        """
        Verify custom ols_fit beta coefficients against numpy's least squares solver.
        """
        custom_beta, _ = ols_fit(self.X_list, self.y_list)
        custom_beta_flat = [b[0] for b in custom_beta]
        
        # rcond=None loai bo FutureWarning va dung do chinh xac cua may
        numpy_beta, residuals, rank, s = np.linalg.lstsq(self.X_arr, self.y_arr, rcond=None)
        numpy_beta_flat = numpy_beta.flatten()
        
        np.testing.assert_almost_equal(custom_beta_flat, numpy_beta_flat, decimal=5)

if __name__ == '__main__':
    unittest.main(argv=['first-arg-is-ignored'], exit=False)

......
----------------------------------------------------------------------
Ran 6 tests in 0.007s

OK


#### Ma trận chiếu - Hat Matrix

Ma trận chiếu được tìm bằng công thức:


$$\mathbf{H} = \mathbf{X}(\mathbf{X}^T\mathbf{X})^{-1}\mathbf{X}^T \in \mathbb{R}^{n \times n}.$$


Tính chất của ma trận H:
* $\mathbf{H}^2 = \mathbf{H}$ (tính lũy đẳng - idempotent).
* $\mathbf{H}^T = \mathbf{H}$ (đối xứng).
* Giá trị riêng của $\mathbf{H}$: chỉ là 0 hoặc 1.
* $\text{rank}(\mathbf{H}) = p + 1$.
* Giá trị fitted: $\hat{\mathbf{y}} = \mathbf{Hy}$; phần dư: $\hat{\boldsymbol{\varepsilon}} = (\mathbf{I} - \mathbf{H})\mathbf{y}$.

In [11]:
# --- Unittest cho ham hat_matrix ---

from ols_implementation import hat_matrix

class TestHatMatrixAgainstLibraries(unittest.TestCase):
    
    def setUp(self):
        # Tao bo du lieu gia lap
        # X bao gom cot intercept la 1.0
        self.X_list = [
            [1.0, 2.5, 1.2],
            [1.0, 1.2, 0.8],
            [1.0, 3.8, 2.5],
            [1.0, 4.1, 3.1],
            [1.0, 5.5, 4.0],
            [1.0, 0.5, 0.1]
        ]
        
        self.X_arr = np.array(self.X_list)

    def test_hat_matrix_against_numpy(self):
        """
        Verify custom hat_matrix output against numpy matrix operations.
        """
        # Lay ket qua tu custom
        custom_H = hat_matrix(self.X_list)
        
        # Lay ket qua tu numpy
        # Cong thuc: H = X * (X^T * X)^-1 * X^T
        X_t = self.X_arr.T
        X_t_X_inv = np.linalg.inv(X_t @ self.X_arr)
        numpy_H = self.X_arr @ X_t_X_inv @ X_t
        
        # Kiem tra cac phan tu ma tran khop toi 5 chu so thap phan
        np.testing.assert_almost_equal(custom_H, numpy_H, decimal=5)

    def test_hat_matrix_properties(self):
        """
        Verify the mathematical properties of the hat matrix: symmetric and idempotent.
        """
        custom_H = hat_matrix(self.X_list)
        
        # Tinh chat 1: Doi xung (H = H^T)
        custom_H_t = mat_trans(custom_H)
        np.testing.assert_almost_equal(custom_H, custom_H_t, decimal=5)
        
        # Tinh chat 2: Dung han (H * H = H)
        custom_H_squared = mat_mul(custom_H, custom_H)
        np.testing.assert_almost_equal(custom_H, custom_H_squared, decimal=5)

if __name__ == '__main__':
    unittest.main(argv=['first-arg-is-ignored'], exit=False)

......
----------------------------------------------------------------------
Ran 6 tests in 0.006s

OK


#### Minh họa định lý Gauss-Markov bằng Monte Carlo

Mô phỏng sử dụng 1000 vòng lặp Monte Carlo để chứng minh thực nghiệm định lý Gauss-Markov, khẳng định OLS là Ước lượng tuyến tính không chệch tốt nhất (BLUE - Best Linear Unbiased Estimator).

Ba yếu tố sau được kiểm chứng:
1. **Tính không chệch:** Chứng minh $\mathbb{E}[\hat{\beta}] = \beta$. So sánh kỳ vọng thực nghiệm của ước lượng OLS (toàn bộ dữ liệu, $n=100$) và một ước lượng tuyến tính khác (sử dụng một phần dữ liệu, $n=70$). Cả hai đều phải hội tụ về $\beta_{true}$.
2. **Tính hiệu quả / Phương sai nhỏ nhất (Best):** Khẳng định $Var(\hat{\beta}_{OLS}) \le Var(\tilde{\beta})$. So sánh phương sai thực nghiệm của hai ước lượng trên. OLS sẽ cho phương sai nhỏ hơn, chứng minh đây là ước lượng hiệu quả nhất.
3. **Đối chiếu lý thuyết:** So sánh phương sai OLS thực nghiệm rút ra từ mô phỏng Monte Carlo với phương sai OLS lý thuyết được tính theo công thức $\sigma^2(X^TX)^{-1}$.

In [12]:
# --- Minh hoa dinh ly Gauss-Markov bang Monte Carlo ---

# Thiet lap tham so mo phong
np.random.seed(42)
n_sims = 1000
n, p = 100, 3
sigma = 1.5

# Tao du lieu mo phong bang NumPy
X_np = np.random.randn(n, p)
X_np = np.c_[np.ones(n), X_np]
beta_true_np = np.array([2.5, 1.2, -0.8, 3.0]).reshape(-1, 1)

# Chuyen doi sang 2D list de su dung voi cac ham tu cai
X_list = X_np.tolist()

# Tao uoc luong tuyen tinh khong chech (LUE) kem hieu qua hon de so sanh
# Bang cach chi su dung 70% luong du lieu
n_sub = int(n * 0.7)
X_sub_list = X_np[:n_sub].tolist()

ols_estimates = []
alt_estimates = []

# Chay Monte Carlo
for _ in range(n_sims):
    # Sinh nhieu va vector y bang NumPy
    eps_np = np.random.normal(0, sigma, (n, 1))
    y_np = X_np @ beta_true_np + eps_np
    
    # Chuyen vector y sang 2D list
    y_list = y_np.tolist()
    y_sub_list = y_np[:n_sub].tolist()
    
    # Uoc luong OLS bang ham tu cai (ols_fit)
    beta_hat_ols, _ = ols_fit(X_list, y_list)
    
    # Uoc luong thay the bang ham tu cai tren tap du lieu nho hon
    beta_hat_alt, _ = ols_fit(X_sub_list, y_sub_list)
    
    # Luu ket qua (chuyen 2D list column vector thanh 1D list)
    ols_estimates.append([b[0] for b in beta_hat_ols])
    alt_estimates.append([b[0] for b in beta_hat_alt])

# Chuyen mang ket qua sang NumPy de tinh toan thong ke
ols_estimates_np = np.array(ols_estimates)
alt_estimates_np = np.array(alt_estimates)

ols_mean = np.mean(ols_estimates_np, axis=0)
alt_mean = np.mean(alt_estimates_np, axis=0)

ols_var = np.var(ols_estimates_np, axis=0)
alt_var = np.var(alt_estimates_np, axis=0)

# Tinh phuong sai ly thuyet bang NumPy de kiem chung
XTX_inv = np.linalg.inv(X_np.T @ X_np)
theoretical_var = (sigma**2) * np.diag(XTX_inv)

# In ket qua
print("--- KIEM CHUNG TINH KHONG CHECH (E[beta] = beta) ---")
print(f"Beta thuc te           : {beta_true_np.flatten()}")
print(f"Ky vong OLS            : {ols_mean}")
print(f"Ky vong uoc luong khac : {alt_mean}")

print("\n--- KIEM CHUNG PHUONG SAI NHO NHAT (BLUE) ---")
print(f"Phuong sai ly thuyet OLS : {theoretical_var}")
print(f"Phuong sai thuc te OLS   : {ols_var}")
print(f"Phuong sai uoc luong khac: {alt_var}")

--- KIEM CHUNG TINH KHONG CHECH (E[beta] = beta) ---
Beta thuc te           : [ 2.5  1.2 -0.8  3. ]
Ky vong OLS            : [ 2.50101355  1.20260482 -0.80261208  2.9944932 ]
Ky vong uoc luong khac : [ 2.50090563  1.19819289 -0.79960851  2.99447599]

--- KIEM CHUNG PHUONG SAI NHO NHAT (BLUE) ---
Phuong sai ly thuyet OLS : [0.02361008 0.03404381 0.02417741 0.01894478]
Phuong sai thuc te OLS   : [0.02571405 0.0372188  0.0236411  0.02012301]
Phuong sai uoc luong khac: [0.03658635 0.05546988 0.03752435 0.02673934]


In [13]:
from ols_implementation import model_metrics, coef_inference, ols_fit
import scipy.stats as stats

class TestInferenceAndMetrics(unittest.TestCase):
    
    def setUp(self):
        self.X_list = [
            [1.0, 2.5, 1.2],
            [1.0, 1.2, 0.8],
            [1.0, 3.8, 2.5],
            [1.0, 4.1, 3.1],
            [1.0, 5.5, 4.0],
            [1.0, 0.5, 0.1]
        ]
        self.y_list = [
            [7.2],
            [4.5],
            [10.1],
            [11.0],
            [14.8],
            [2.1]
        ]
        self.X_arr = np.array(self.X_list)
        self.y_arr = np.array(self.y_list).flatten()
        self.n = len(self.X_list)
        self.p = len(self.X_list[0]) - 1
        
        # Pre-calculate beta and sigma2 for tests
        self.beta, self.sigma2 = ols_fit(self.X_list, self.y_list)
        self.y_hat = [[sum(self.X_list[i][j] * self.beta[j][0] for j in range(self.p+1))] for i in range(self.n)]

    def test_model_metrics_against_numpy(self):
        """
        Verify RSS, TSS, R-squared, and F-statistic against numpy/scipy manual calculations.
        """
        metrics = model_metrics(self.y_list, self.y_hat, self.p)
        
        # Reference calculations
        y_flat = self.y_arr
        y_hat_flat = np.array(self.y_hat).flatten()
        rss_ref = np.sum((y_flat - y_hat_flat)**2)
        tss_ref = np.sum((y_flat - np.mean(y_flat))**2)
        r2_ref = 1 - rss_ref / tss_ref
        f_stat_ref = ((tss_ref - rss_ref) / self.p) / (rss_ref / (self.n - self.p - 1))
        
        self.assertAlmostEqual(metrics['RSS'], rss_ref, places=5)
        self.assertAlmostEqual(metrics['TSS'], tss_ref, places=5)
        self.assertAlmostEqual(metrics['R_squared'], r2_ref, places=5)
        self.assertAlmostEqual(metrics['F_statistic'], f_stat_ref, places=5)

    def test_coef_inference_against_numpy(self):
        """
        Verify Standard Errors and t-statistics against numpy/scipy manual calculations.
        """
        inference = coef_inference(self.X_list, self.y_list, self.beta, self.sigma2)
        
        # Reference calculations
        C = np.linalg.inv(self.X_arr.T @ self.X_arr)
        se_ref = np.sqrt(self.sigma2 * np.diag(C))
        beta_flat = np.array(self.beta).flatten()
        t_ref = beta_flat / se_ref
        p_ref = 2 * (1 - stats.t.cdf(np.abs(t_ref), self.n - self.p - 1))
        
        np.testing.assert_almost_equal(inference['Standard_Errors'], se_ref, decimal=5)
        np.testing.assert_almost_equal(inference['t_statistics'], t_ref, decimal=5)
        np.testing.assert_almost_equal(inference['p_values'], p_ref, decimal=5)

if __name__ == '__main__':
    unittest.main(argv=['first-arg-is-ignored'], exit=False)

......
----------------------------------------------------------------------
Ran 6 tests in 0.005s

OK
